# 🌦️ Weather Image Classification (TensorFlow)

![TensorFlow](https://img.shields.io/badge/TensorFlow-2.x-orange?logo=tensorflow)
![Keras](https://img.shields.io/badge/Keras-EfficientNetB0-red?logo=keras)
![Python](https://img.shields.io/badge/Python-3.10+-blue?logo=python)
![License](https://img.shields.io/badge/License-MIT-green)
![Status](https://img.shields.io/badge/Status-Portfolio%20Project-brightgreen)

A complete Machine Learning project presenting a weather-phenomenon
classifier based on a single photograph, using **Transfer Learning**,
**Fine Tuning**, and **Grad-CAM** (model explainability) on the
**EfficientNetB0** architecture (TensorFlow / Keras).

The notebook's structure and methodology are designed to guide the reader step by step through every stage of the project, providing a clear rationale for each design decision—from data preparation to model deployment.

---

## 🗂️ Table of Contents

1. [Problem Statement](#problem)
2. [Dataset Description](#dataset)
3. [Library Imports](#importy)
4. [TensorFlow Configuration](#konfiguracja)
5. [Dataset Download](#pobieranie)
6. [Data Analysis](#analiza)
7. [Class Visualization](#wizualizacja)
8. [Data Augmentation](#augmentacja)
9. [Creating Train / Validation / Test Sets](#podzial)
10. [Model Architecture](#architektura)
11. [Transfer Learning](#transfer-learning)
12. [Fine Tuning](#fine-tuning)
13. [Training](#trenowanie)
14. [Accuracy](#accuracy)
15. [Loss](#loss)
16. [Precision](#precision)
17. [Recall](#recall)
18. [F1-score](#f1)
19. [Confusion Matrix](#confusion-matrix)
20. [Classification Report](#classification-report)
21. [Single Image Prediction](#pred-single)
22. [Multiple Image Prediction](#pred-multi)
23. [Misclassification Visualization](#bledy)
24. [Grad-CAM — What the Model Is Looking At](#gradcam)
25. [Saving the Model](#zapis)
26. [Loading the Model](#wczytanie)
27. [Prediction on New Images](#nowe-zdjecia)
28. [Summary](#podsumowanie)
29. [Limitations](#ograniczenia)
30. [Possible Improvements](#ulepszenia)

---

<a name="problem"></a>
## 1️⃣ Problem Statement

Automatically recognizing weather phenomena from a photograph has practical
applications in areas such as: camera-based weather monitoring systems, safe
navigation for autonomous vehicles (detecting hazardous conditions — glaze
ice, fog, sandstorms), mobile apps for automatic photo tagging, and support
for early-warning systems.

**Task type:** multi-class image classification — each photograph belongs to
exactly one of **11 classes** of weather phenomena.

**Input:** a single photograph (RGB), not necessarily with a standardized
framing or lighting (unlike laboratory-condition photos).

**Output:** a class label (e.g. `rainbow`) together with a prediction
confidence level (a probability from the `softmax` layer).

**Success metric:** accuracy and F1-score (macro) on the test set.
Additionally, since weather phenomena can look visually similar (e.g. `frost`
vs. `glaze`, `snow` vs. `rime`), we track the confusion matrix to identify
specific class pairs the model struggles with.

<a name="dataset"></a>
## 2️⃣ Dataset Description

This project uses the **Weather Dataset** ([Kaggle —
jehanbhathena/weather-dataset](https://www.kaggle.com/datasets/jehanbhathena/weather-dataset)).

| Feature | Value |
|---|---|
| Number of classes | 11 |
| Number of images | ~6862 |
| Classes | `dew`, `fogsmog`, `frost`, `glaze`, `hail`, `lightning`, `rain`, `rainbow`, `rime`, `sandstorm`, `snow` |
| Format | color RGB images, varying source resolution |
| Photo conditions | "real-world" images (sourced from the internet) — varying background, framing, lighting |
| Train/val/test split | none in the original dataset — we create our own (Section 9) |

Unlike datasets photographed under controlled laboratory conditions, this
dataset contains images gathered from various internet sources — which more
closely reflects the real-world conditions in which the model might actually
be used, but also introduces greater visual diversity within the same class.


<a name="importy"></a>
## 3️⃣ Library Imports


In [ ]:
# ---------------------------------------------------------------
# Installation (if running in Google Colab / a fresh environment)
# ---------------------------------------------------------------
# IMPORTANT: we deliberately do NOT use --upgrade for tensorflow/pandas/scikit-learn.
# Google Colab ships with a pre-installed, mutually compatible set of these
# libraries. Forcing the latest versions can desynchronize that set and trigger
# a cascade of dependency conflicts. We only install what is actually missing.
!pip install --quiet kaggle seaborn

import os
import json
import datetime
import pathlib
import random
import getpass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Plot style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ TensorFlow version:", tf.__version__)


<a name="konfiguracja"></a>
## 4️⃣ TensorFlow Configuration

We check GPU availability and configure **dynamic memory allocation**
(`memory growth`) so TensorFlow doesn't immediately reserve the entire GPU
memory — good practice, especially on a shared environment (e.g. Google
Colab). This section also defines the global configuration constants used
throughout the notebook.


In [ ]:
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Detected {len(gpus)} GPU(s). Memory growth enabled.")
        for gpu in gpus:
            print("   -", gpu)
    except RuntimeError as e:
        print("⚠️ Failed to configure memory growth:", e)
else:
    print("⚠️ No GPU detected — training will run on CPU (significantly slower).")
    print("   In Google Colab: Runtime → Change runtime type → Hardware accelerator: GPU")

# Mixed precision (optional) — speeds up training on newer GPUs (Tensor Cores):
# from tensorflow.keras import mixed_precision
# mixed_precision.set_global_policy("mixed_float16")
# print("✅ Mixed precision enabled:", mixed_precision.global_policy())

# ---------------------------------------------------------------
# Global project configuration constants
# ---------------------------------------------------------------
IMG_SIZE = 224          # required input size for EfficientNetB0
BATCH_SIZE = 32
MODELS_DIR = "models"
RESULTS_DIR = "results"
PREDICTIONS_DIR = "predictions"
LOGS_DIR = "logs"

for directory in (MODELS_DIR, RESULTS_DIR, PREDICTIONS_DIR, LOGS_DIR):
    os.makedirs(directory, exist_ok=True)

print(f"🖼️ IMG_SIZE = {IMG_SIZE}, BATCH_SIZE = {BATCH_SIZE}")


<a name="pobieranie"></a>
## 5️⃣ Dataset Download

We download the **Weather Dataset** from Kaggle using the official `kaggle`
client. This requires your own (free) API key:

1. Create an account at [kaggle.com](https://www.kaggle.com) (if you don't
   have one yet).
2. **Account → API → Create New API Token** — this downloads a `kaggle.json`
   file containing your `username` and `key`.
3. Paste the values from that file below when the notebook prompts for them
   (`getpass` — the key is **never** hard-coded anywhere.

Since the exact folder structure inside the Kaggle archive can vary (different
nesting levels), the code below **automatically detects** the correct folder
that directly contains the class subfolders — instead of hard-coding a
specific class count or folder name. This makes the code work correctly
regardless of the archive variant.


In [ ]:
def find_data_directory(root_dir):
    # Looks for a folder whose direct subfolders are ALL "leaves" (i.e. they
    # contain files, not further subfolders) — meaning they are class folders
    # containing images. Among the candidates, picks the one with the largest
    # total file count (the most likely candidate for the actual dataset).
    candidates = []
    for dirpath, dirnames, _ in os.walk(root_dir):
        if not dirnames:
            continue
        subdirs = [os.path.join(dirpath, d) for d in dirnames]
        all_leaves = all(
            not any(os.path.isdir(os.path.join(sd, x)) for x in os.listdir(sd))
            for sd in subdirs
        )
        if all_leaves and len(dirnames) > 1:
            total_files = sum(len(os.listdir(sd)) for sd in subdirs)
            candidates.append((dirpath, total_files))

    if not candidates:
        raise FileNotFoundError(
            f"No folder with class subfolders was found in '{root_dir}'.\n"
            f"Check the downloaded data structure manually, e.g.:\n"
            f"  !find {root_dir} -maxdepth 3 -type d\n"
            f"and set the DATA_DIR variable manually to the correct path."
        )

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0]


print("🔑 Downloading the dataset requires a Kaggle API key (Account → API → Create New API Token).")
KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME") or getpass.getpass("Enter KAGGLE_USERNAME: ")
KAGGLE_KEY = os.environ.get("KAGGLE_KEY") or getpass.getpass("Enter KAGGLE_KEY: ")
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = KAGGLE_KEY

DATA_ROOT = "data"
KAGGLE_DATASET = "jehanbhathena/weather-dataset"
DOWNLOAD_MARKER = os.path.join(DATA_ROOT, ".download_complete")

os.makedirs(DATA_ROOT, exist_ok=True)

if not os.path.exists(DOWNLOAD_MARKER):
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_ROOT} --unzip
    pathlib.Path(DOWNLOAD_MARKER).touch()
    print("✅ Download complete.")
else:
    print("✅ Dataset already downloaded — skipping re-download.")

DATA_DIR = find_data_directory(DATA_ROOT)
print(f"📁 Detected data folder: {DATA_DIR}")

class_names = sorted(
    d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))
)
NUM_CLASSES = len(class_names)
TOTAL_EXAMPLES = sum(len(os.listdir(os.path.join(DATA_DIR, c))) for c in class_names)

print(f"🔢 Number of classes: {NUM_CLASSES}")
print(f"🖼️ Number of images: {TOTAL_EXAMPLES}")
print(f"\nClasses: {class_names}")

assert NUM_CLASSES == 11, (
    f"Expected 11 classes, detected {NUM_CLASSES}. Check the downloaded data "
    f"structure — a different dataset variant than expected may have been downloaded."
)


<a name="analiza"></a>
## 6️⃣ Data Analysis

We check the distribution of image counts across classes. Imbalanced data can
cause the model to "favor" more populous classes — worth diagnosing this
right from the start.


In [ ]:
class_counts = pd.DataFrame({
    "class": class_names,
    "image_count": [len(os.listdir(os.path.join(DATA_DIR, c))) for c in class_names],
}).sort_values("image_count", ascending=True)

print(class_counts.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.barh(class_counts["class"], class_counts["image_count"], color="steelblue")
plt.xlabel("Number of images")
plt.title("Image count per class — Weather Dataset (11 classes)")
plt.tight_layout()
plt.show()

imbalance_ratio = class_counts["image_count"].max() / class_counts["image_count"].min()
print(f"\n📊 Imbalance ratio (max/min): {imbalance_ratio:.1f}x")


### Building the File List

We build a single list containing the path and label for **every** image in
the dataset — the starting point for visualization (Section 7) and the
formal train/validation/test split (Section 9).


In [ ]:
image_paths = []
image_labels = []

for class_idx, class_name in enumerate(class_names):
    class_dir = os.path.join(DATA_DIR, class_name)
    for fname in os.listdir(class_dir):
        image_paths.append(os.path.join(class_dir, fname))
        image_labels.append(class_idx)

image_paths = np.array(image_paths)
image_labels = np.array(image_labels)

print(f"🖼️ Total number of images: {len(image_paths)}")


<a name="wizualizacja"></a>
## 7️⃣ Class Visualization

We display one example image from **each** of the 11 classes — this helps
build intuition about the visual characteristics of each weather phenomenon
and spot potential similarities between classes (e.g. `frost` and `glaze`, or
`snow` and `rime`) that could be challenging for the model.


In [ ]:
def load_image(path):
    raw = tf.io.read_file(path)
    image = tf.io.decode_image(raw, channels=3, expand_animations=False)
    return image


rng = np.random.default_rng(SEED)

plt.figure(figsize=(15, 12))
for i, class_name in enumerate(class_names):
    class_indices = np.where(image_labels == i)[0]
    idx = rng.choice(class_indices)
    image = load_image(image_paths[idx])
    ax = plt.subplot(3, 4, i + 1)
    plt.imshow(image.numpy())
    plt.title(class_name, fontsize=11)
    plt.axis("off")
plt.suptitle("Example image from each class — Weather Dataset", fontsize=15)
plt.tight_layout()
plt.show()


<a name="augmentacja"></a>
## 8️⃣ Data Augmentation

**Data Augmentation** is a technique for artificially increasing the
diversity of training data through random image transformations — it reduces
the risk of overfitting and improves the model's ability to generalize.

We implement augmentation as **Keras layers** (`tf.keras.Sequential`),
attached directly to the model — active only during training
(`training=True`), automatically disabled during evaluation and prediction.

> ℹ️ **Dataset-specific note:** we deliberately **don't** apply
> `RandomFlip("horizontal")` —
> rather, we apply it cautiously, because phenomena such as `rainbow` or
> `lightning` have a distinctive, though not necessarily symmetric, shape.
> A horizontal flip still makes sense, though, since left-right orientation
> carries no diagnostic information for any of the 11 classes.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
    layers.RandomBrightness(0.15),
], name="data_augmentation")

# Visualization: the same image+augmentation pair applied 9 times
sample_idx_aug = rng.integers(0, len(image_paths))
sample_image = tf.image.resize(load_image(image_paths[sample_idx_aug]), (IMG_SIZE, IMG_SIZE))
sample_label = image_labels[sample_idx_aug]
sample_image_batch = tf.expand_dims(sample_image, axis=0)

plt.figure(figsize=(12, 12))
for i in range(9):
    augmented = data_augmentation(sample_image_batch, training=True)
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(tf.clip_by_value(augmented[0], 0, 255).numpy().astype("uint8"))
    plt.axis("off")
plt.suptitle(f"Data Augmentation — example: {class_names[sample_label]}", fontsize=13)
plt.tight_layout()
plt.show()


<a name="podzial"></a>
## 9️⃣ Creating Train / Validation / Test Sets

We split the list of all images into three disjoint parts — explicitly and
reproducibly (fixed `SEED`):

| Set | Share | Purpose |
|---|---|---|
| Train | 70% | training the model's weights |
| Validation | 15% | monitoring during training (early stopping, hyperparameter selection) |
| Test | 15% | final, one-time model evaluation |

We build a `tf.data` pipeline: reading a file from disk, decoding, resizing
to 224×224, batching, and `prefetch`.


In [ ]:
indices = np.arange(len(image_paths))
rng.shuffle(indices)

n_total = len(indices)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]


def make_dataset_from_indices(idx):
    return tf.data.Dataset.from_tensor_slices((image_paths[idx], image_labels[idx]))


def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = tf.cast(image, tf.float32)  # EfficientNet has built-in normalization — no manual scaling needed
    return image, label


def prepare_dataset(ds, shuffle=False, batch_size=BATCH_SIZE):
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=2000, seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = prepare_dataset(make_dataset_from_indices(train_idx), shuffle=True)
val_ds = prepare_dataset(make_dataset_from_indices(val_idx))
test_ds = prepare_dataset(make_dataset_from_indices(test_idx))

split_sizes = {"train": len(train_idx), "validation": len(val_idx), "test": len(test_idx)}
print("📊 Split sizes:", split_sizes)

plt.figure(figsize=(6, 5))
plt.bar(split_sizes.keys(), split_sizes.values(), color=["#4C72B0", "#DD8452", "#55A868"])
plt.title("Dataset split: Train / Validation / Test")
plt.ylabel("Number of images")
for i, (k, v) in enumerate(split_sizes.items()):
    plt.text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
plt.show()


<a name="architektura"></a>
## 🔟 Model Architecture

**EfficientNetB0** is a convolutional network designed for an optimal
accuracy-to-parameter-count ratio. A good base choice for transfer learning:
powerful enough, yet lightweight and fast to train.

**Final model architecture:**

```
Input (224, 224, 3)
        ↓
Data Augmentation (active only during training)
        ↓
EfficientNetB0 (base, ImageNet weights, initially frozen)
        ↓
GlobalAveragePooling2D
        ↓
Dropout (0.3)
        ↓
Dense (11, softmax)  ← our classification layer
```


In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    pooling=None,
)
base_model.trainable = False  # freeze weights during the transfer-learning stage

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_layer")
x = data_augmentation(inputs)
# training=False: even after unfreezing layers during fine-tuning (Section 12),
# BatchNormalization layers will remain in inference mode — recommended Keras
# practice when fine-tuning models with BatchNorm (prevents "corrupting" the
# learned normalization statistics with small batches during fine-tuning).
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name="global_average_pooling")(x)
x = layers.Dropout(0.3, name="top_dropout")(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="output_layer")(x)

model = tf.keras.Model(inputs, outputs, name="weather_classifier_efficientnetb0")
model.summary()

print(f"\n🔢 Number of layers in the base model: {len(base_model.layers)}")
print(f"🔒 Base model frozen: {not base_model.trainable}")


<a name="transfer-learning"></a>
## 1️⃣1️⃣ Transfer Learning (Phase 1 — Training the Classification Head)

In the first training phase, EfficientNetB0's weights remain **frozen** —
only the newly added classification head is trained. This makes training
fast, and we avoid the risk of destroying the well-trained, general visual
features learned on ImageNet.

**Callbacks used during training:**

| Callback | Role |
|---|---|
| `ModelCheckpoint` | saves the best model version (by `val_accuracy`) |
| `EarlyStopping` | stops training once the model stops improving |
| `ReduceLROnPlateau` | reduces the learning rate when training plateaus |
| `TensorBoard` | logs training progress for visualization |


In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

phase1_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "phase1_feature_extraction.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, min_lr=1e-7, verbose=1
    ),
    callbacks.TensorBoard(log_dir=os.path.join(LOGS_DIR, f"phase1_{timestamp}")),
]

EPOCHS_PHASE1 = 10

history_1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=phase1_callbacks,
    verbose=1,
)


<a name="fine-tuning"></a>
## 1️⃣2️⃣ Fine Tuning (Phase 2 — Precise Adjustment of Base Layers)

After the classification head has stabilized, we **unfreeze** the last
layers of EfficientNetB0, so the model can gently adapt high-level visual
features to the specifics of our dataset.

**Key rules for safe fine-tuning:**

* Only unfreeze the **last N layers** (not the whole model).
* Use a **much smaller learning rate** (`1e-5` instead of `1e-3`).
* `BatchNormalization` layers remain in inference mode regardless of
  unfreezing (see the comment in Section 10).


In [ ]:
FINE_TUNE_AT = len(base_model.layers) - 30  # unfreeze the last 30 layers

base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

trainable_count = sum(1 for layer in base_model.layers if layer.trainable)
print(f"🔓 Unfrozen base-model layers: {trainable_count} / {len(base_model.layers)}")

# Recompilation is required for the `trainable` change to take effect
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

phase2_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=os.path.join(MODELS_DIR, "phase2_fine_tuning.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.2, patience=3, min_lr=1e-8, verbose=1
    ),
    callbacks.TensorBoard(log_dir=os.path.join(LOGS_DIR, f"phase2_{timestamp}")),
]

EPOCHS_PHASE2 = 10
TOTAL_EPOCHS = EPOCHS_PHASE1 + EPOCHS_PHASE2

history_2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history_1.epoch[-1] + 1,
    callbacks=phase2_callbacks,
    verbose=1,
)


<a name="trenowanie"></a>
## 1️⃣3️⃣ Training — Summary of Both Phases

We merge the training history from **Phase 1** (transfer learning) and
**Phase 2** (fine-tuning) into one consistent chart, with a dashed line
marking the transition between phases.


In [ ]:
def merge_histories(hist1, hist2):
    merged = {}
    for key in hist1.history:
        merged[key] = hist1.history[key] + hist2.history[key]
    return merged


history = merge_histories(history_1, history_2)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(history["accuracy"], label="Train accuracy")
axes[0].plot(history["val_accuracy"], label="Validation accuracy")
axes[0].axvline(EPOCHS_PHASE1 - 1, color="gray", linestyle="--", label="Fine-tuning start")
axes[0].set_title("Accuracy — both training phases")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["loss"], label="Train loss")
axes[1].plot(history["val_loss"], label="Validation loss")
axes[1].axvline(EPOCHS_PHASE1 - 1, color="gray", linestyle="--", label="Fine-tuning start")
axes[1].set_title("Loss — both training phases")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "training_curves.png"), dpi=150)
plt.show()

print(f"📈 Best validation accuracy: {max(history['val_accuracy']):.4f}")
print(f"📉 Lowest validation loss:  {min(history['val_loss']):.4f}")


### Preparing Test-Set Predictions

The cell below is run once and used across all subsequent sections
(Accuracy, Precision, Recall, F1, Confusion Matrix, Classification Report,
prediction visualizations) — we collect predictions, ground-truth labels, and
the images themselves (for later visualizations and Grad-CAM).


In [ ]:
y_true = []
y_pred = []
y_pred_probs = []
test_images = []

for images_batch, labels_batch in test_ds:
    preds_batch = model.predict(images_batch, verbose=0)
    y_pred_probs.extend(preds_batch)
    y_pred.extend(np.argmax(preds_batch, axis=1))
    y_true.extend(labels_batch.numpy())
    test_images.extend(images_batch.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_pred_probs = np.array(y_pred_probs)

test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)
print(f"\n📊 Test set results:")
print(f"   Loss:     {test_loss:.4f}")
print(f"   Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)")
print(f"\n✅ Collected predictions for {len(y_true)} test images.")


<a name="accuracy"></a>
## 1️⃣4️⃣ Accuracy

Accuracy is the fraction of correctly classified images out of all images.


In [ ]:
final_test_accuracy = (y_pred == y_true).mean()

plt.figure(figsize=(10, 5))
plt.plot(history["accuracy"], label="Train accuracy")
plt.plot(history["val_accuracy"], label="Validation accuracy")
plt.axhline(final_test_accuracy, color="green", linestyle=":", label=f"Test accuracy ({final_test_accuracy:.3f})")
plt.title("Accuracy — train / validation / test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"🎯 Test accuracy: {final_test_accuracy:.4f} ({final_test_accuracy * 100:.2f}%)")


<a name="loss"></a>
## 1️⃣5️⃣ Loss

The value of the loss function (`sparse_categorical_crossentropy`). A large
gap between `loss` (training) and `val_loss` (validation) would signal
overfitting.


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(history["loss"], label="Train loss")
plt.plot(history["val_loss"], label="Validation loss")
plt.axhline(test_loss, color="green", linestyle=":", label=f"Test loss ({test_loss:.3f})")
plt.title("Loss — train / validation / test")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"📉 Test loss: {test_loss:.4f}")


<a name="precision"></a>
## 1️⃣6️⃣ Precision

**Precision** answers the question: *of the images the model assigned to a
given class, what fraction actually belonged to it?* We compute both the
**macro** variant (unweighted average across classes) and the **weighted**
variant (accounts for class frequency).


In [ ]:
precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
precision_weighted = precision_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"🎯 Precision (macro):    {precision_macro:.4f}")
print(f"🎯 Precision (weighted): {precision_weighted:.4f}")


<a name="recall"></a>
## 1️⃣7️⃣ Recall

**Recall** answers the question: *of all the images that actually belong to a
given class, what fraction did the model correctly identify?* Important e.g.
for classes tied to hazardous phenomena (`hail`, `lightning`, `sandstorm`) —
missing such a phenomenon can have more serious consequences than a false
alarm.


In [ ]:
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)
recall_weighted = recall_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"🎯 Recall (macro):    {recall_macro:.4f}")
print(f"🎯 Recall (weighted): {recall_weighted:.4f}")


<a name="f1"></a>
## 1️⃣8️⃣ F1-score

**F1-score** is the harmonic mean of precision and recall — a single metric
balancing both aspects, useful for imbalanced data.


In [ ]:
f1_macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
f1_weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"🎯 F1-score (macro):    {f1_macro:.4f}")
print(f"🎯 F1-score (weighted): {f1_weighted:.4f}")

metrics_summary = pd.DataFrame({
    "metric": ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"],
    "value": [final_test_accuracy, precision_macro, recall_macro, f1_macro],
})
print("\n", metrics_summary.to_string(index=False))

metrics_summary.to_csv(os.path.join(RESULTS_DIR, "metrics_summary.csv"), index=False)
print(f"\n💾 Metrics summary saved to {RESULTS_DIR}/metrics_summary.csv")


<a name="confusion-matrix"></a>
## 1️⃣9️⃣ Confusion Matrix

The confusion matrix shows exactly which classes are being confused with each
other. With 11 classes, the matrix is fully readable with numeric annotations
in every cell.


In [ ]:
conf_matrix = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(11, 9))
sns.heatmap(
    conf_matrix,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    cbar=True,
)
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.title("Confusion Matrix — test set")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

print(f"💾 Confusion matrix saved to {RESULTS_DIR}/confusion_matrix.png")

conf_matrix_off_diagonal = conf_matrix.copy()
np.fill_diagonal(conf_matrix_off_diagonal, 0)
top_confusions_idx = np.dstack(
    np.unravel_index(np.argsort(-conf_matrix_off_diagonal.ravel())[:5], conf_matrix.shape)
)[0]

print("\n🔀 Most frequently confused class pairs:")
for true_idx, pred_idx in top_confusions_idx:
    count = conf_matrix[true_idx, pred_idx]
    if count > 0:
        print(f"   {class_names[true_idx]}  →  {class_names[pred_idx]}  ({count} cases)")


<a name="classification-report"></a>
## 2️⃣0️⃣ Classification Report

The full `scikit-learn` report with **per-class** metrics (precision, recall,
f1-score, support).


In [ ]:
report_dict = classification_report(
    y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose()

print(classification_report(y_true, y_pred, target_names=class_names, zero_division=0))

report_df.to_csv(os.path.join(RESULTS_DIR, "classification_report.csv"))
print(f"💾 Full report saved to {RESULTS_DIR}/classification_report.csv")

worst_classes = report_df.iloc[:NUM_CLASSES].sort_values("f1-score").head(3)
print("\n⚠️ Classes with the lowest F1-score:")
print(worst_classes[["precision", "recall", "f1-score", "support"]])


<a name="pred-single"></a>
## 2️⃣1️⃣ Single Image Prediction

The helper function `predict_single_image` takes a single image, runs it
through the model, and returns the predicted class along with the confidence
level.


In [ ]:
def predict_single_image(model, image, class_names, true_label=None):
    # Predicts the class for a single image and displays the result.
    image_batch = tf.expand_dims(image, axis=0)
    probs = model.predict(image_batch, verbose=0)[0]
    pred_idx = np.argmax(probs)
    confidence = probs[pred_idx]

    plt.figure(figsize=(5, 5))
    plt.imshow(image.numpy().astype("uint8") if hasattr(image, "numpy") else image.astype("uint8"))
    title = f"Prediction: {class_names[pred_idx]}\nConfidence: {confidence * 100:.1f}%"
    if true_label is not None:
        title = f"True: {class_names[true_label]}\n" + title
    plt.title(title, fontsize=10)
    plt.axis("off")
    plt.show()

    return class_names[pred_idx], confidence


random_idx = random.randint(0, len(test_images) - 1)
predicted_class, confidence = predict_single_image(
    model, tf.constant(test_images[random_idx]), class_names, true_label=y_true[random_idx]
)


<a name="pred-multi"></a>
## 2️⃣2️⃣ Multiple Image Prediction

A grid of predictions for several images at once — the title is green for a
correct prediction and red for an incorrect one.


In [ ]:
def plot_prediction_grid(images, y_true_arr, y_pred_arr, y_pred_probs_arr, class_names, n=9):
    indices = random.sample(range(len(images)), n)
    plt.figure(figsize=(14, 14))
    for i, idx in enumerate(indices):
        ax = plt.subplot(int(np.sqrt(n)), int(np.sqrt(n)), i + 1)
        plt.imshow(images[idx].astype("uint8"))
        correct = y_true_arr[idx] == y_pred_arr[idx]
        color = "green" if correct else "red"
        confidence = y_pred_probs_arr[idx][y_pred_arr[idx]] * 100
        plt.title(
            f"Pred: {class_names[y_pred_arr[idx]]}\n({confidence:.0f}%)",
            fontsize=9, color=color,
        )
        plt.axis("off")
    plt.suptitle("Predictions on multiple images (green = correct, red = incorrect)", fontsize=13)
    plt.tight_layout()
    plt.show()


plot_prediction_grid(test_images, y_true, y_pred, y_pred_probs, class_names, n=9)


<a name="bledy"></a>
## 2️⃣3️⃣ Misclassification Visualization

A deliberate look at the model's specific mistakes — helps assess whether the
errors are "understandable" (e.g. visually similar weather phenomena, like
`frost` vs. `glaze`) or point to a data or model issue.


In [ ]:
misclassified_idx = np.where(y_true != y_pred)[0]
print(f"❌ Number of misclassifications: {len(misclassified_idx)} / {len(y_true)} "
      f"({len(misclassified_idx) / len(y_true) * 100:.2f}%)")

n_show = min(9, len(misclassified_idx))
sample_errors = np.random.choice(misclassified_idx, n_show, replace=False)

plt.figure(figsize=(14, 14))
for i, idx in enumerate(sample_errors):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(test_images[idx].astype("uint8"))
    confidence = y_pred_probs[idx][y_pred[idx]] * 100
    plt.title(
        f"True: {class_names[y_true[idx]]}\n"
        f"Predicted: {class_names[y_pred[idx]]} ({confidence:.0f}%)",
        fontsize=9, color="red",
    )
    plt.axis("off")
plt.suptitle("Examples of misclassifications", fontsize=13)
plt.tight_layout()
plt.show()


<a name="gradcam"></a>
## 2️⃣4️⃣ Grad-CAM — What the Model Is Looking At

**Grad-CAM** (Gradient-weighted Class Activation Mapping) is an
explainability (*Explainable AI*) technique that shows **which regions of the
image the model actually focused on** when making its classification
decision. It works by analyzing the gradients of the chosen class with
respect to the activations of the last convolutional layer — regions with a
large influence on the prediction are highlighted on a "heatmap" (red = high
influence, blue = low influence).

This is an important diagnostic tool: it lets us verify whether the model is
genuinely "looking" at regions associated with the given weather phenomenon
(e.g. raindrops, snowflakes), or whether it's instead basing its prediction
on incidental background features (so-called *shortcut learning*).


In [ ]:
def find_last_conv_layer(keras_model):
    # Finds the name of the last convolutional layer (4D output: batch,
    # height, width, channels) in the given model, searching from the end.
    for layer in reversed(keras_model.layers):
        if len(layer.output.shape) == 4:
            return layer.name
    raise ValueError("No convolutional layer found in the given model.")


LAST_CONV_LAYER_NAME = find_last_conv_layer(base_model)
print(f"🎯 Last convolutional layer of EfficientNetB0: '{LAST_CONV_LAYER_NAME}'")

# We build a separate, standalone Grad-CAM model based EXCLUSIVELY on
# `base_model`'s own input/output (not on `model`, in which base_model is
# nested as a single layer). This matters: in newer Keras versions (Keras 3),
# reading the `.output` of a layer located inside a model that is itself
# nested inside another model can be unstable and raise errors like "Graph
# disconnected" — we sidestep this issue by reconstructing the classification
# head (the same, already-trained layers) directly on top of `base_model`'s
# output, creating a fully non-nested computation graph.
gap_layer = model.get_layer("global_average_pooling")
dropout_layer = model.get_layer("top_dropout")
output_layer = model.get_layer("output_layer")

conv_output_tensor = base_model.get_layer(LAST_CONV_LAYER_NAME).output
head_output_tensor = output_layer(dropout_layer(gap_layer(base_model.output), training=False))

grad_model = tf.keras.Model(
    inputs=base_model.input,
    outputs=[conv_output_tensor, head_output_tensor],
)


def make_gradcam_heatmap(img_array, grad_model, pred_index=None):
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)


def overlay_gradcam(image, heatmap, alpha=0.4):
    # Overlays the Grad-CAM heatmap ('jet' colormap) on the original image.
    image_uint8 = np.uint8(np.clip(image, 0, 255))
    heatmap_uint8 = np.uint8(255 * heatmap)

    jet = cm.get_cmap("jet")
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_uint8]
    jet_heatmap = tf.image.resize(jet_heatmap, (image_uint8.shape[0], image_uint8.shape[1]))
    jet_heatmap = np.uint8(jet_heatmap.numpy() * 255)

    superimposed = np.uint8(np.clip(jet_heatmap * alpha + image_uint8 * (1 - alpha), 0, 255))
    return superimposed


def show_gradcam(grad_model, image, class_names, true_label=None):
    img_array = tf.expand_dims(image, axis=0)
    heatmap, pred_idx = make_gradcam_heatmap(img_array, grad_model)
    overlay = overlay_gradcam(image.numpy() if hasattr(image, "numpy") else image, heatmap)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(np.uint8(image.numpy() if hasattr(image, "numpy") else image))
    axes[0].set_title("Original image")
    axes[0].axis("off")

    axes[1].imshow(heatmap, cmap="jet")
    axes[1].set_title("Activation map (Grad-CAM)")
    axes[1].axis("off")

    axes[2].imshow(overlay)
    title = f"Overlaid map\nPrediction: {class_names[pred_idx]}"
    if true_label is not None:
        title = f"True: {class_names[true_label]}\n" + title
    axes[2].set_title(title)
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()
    return overlay


# Example on a randomly chosen, correctly classified image
correct_idx = np.where(y_true == y_pred)[0]
example_idx = np.random.choice(correct_idx)
_ = show_gradcam(
    grad_model, tf.constant(test_images[example_idx]), class_names, true_label=y_true[example_idx]
)


<a name="zapis"></a>
## 2️⃣5️⃣ Saving the Model

We save the final model in Keras's native format (`.keras`) along with the
list of class names in a separate JSON file — required to interpret the
results after loading the model in another environment.


In [ ]:
FINAL_MODEL_PATH = os.path.join(MODELS_DIR, "weather_classifier_efficientnetb0_final.keras")
CLASS_NAMES_PATH = os.path.join(MODELS_DIR, "class_names.json")

model.save(FINAL_MODEL_PATH)
print(f"💾 Model saved: {FINAL_MODEL_PATH}")

with open(CLASS_NAMES_PATH, "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)
print(f"💾 Class names saved: {CLASS_NAMES_PATH}")

model_size_mb = os.path.getsize(FINAL_MODEL_PATH) / (1024 * 1024)
print(f"📦 Model file size: {model_size_mb:.1f} MB")


<a name="wczytanie"></a>
## 2️⃣6️⃣ Loading the Model

We verify that the saved model loads correctly and produces identical results
to the model kept in memory.


In [ ]:
loaded_model = tf.keras.models.load_model(FINAL_MODEL_PATH)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8") as f:
    loaded_class_names = json.load(f)

loaded_test_loss, loaded_test_accuracy = loaded_model.evaluate(test_ds, verbose=0)

print(f"✅ Model loaded successfully.")
print(f"   Accuracy (original model): {test_accuracy:.4f}")
print(f"   Accuracy (loaded model):   {loaded_test_accuracy:.4f}")
assert abs(test_accuracy - loaded_test_accuracy) < 1e-4, "Mismatch between the original and loaded model!"
print("✅ Results are identical — the model is ready for deployment.")


<a name="nowe-zdjecia"></a>
## 2️⃣7️⃣ Prediction on New Images

A full "file-to-result" inference pipeline — with automatic saving of the
prediction result (image + label + confidence) to the `predictions/` folder,
simulating how results might be archived in a real application.

**How to actually use this function on your own image — three ways:**

1. **Google Colab (easiest):** run the cell below — a "Choose Files" button
   will appear, letting you pick an image from your computer. It will be
   automatically uploaded to the notebook and classified.
2. **Local Jupyter / any environment:** upload your image to the `images/`
   folder in the repository, then in a new cell call:
   ```python
   predict_new_image("images/my_photo.jpg", loaded_model, loaded_class_names)
   ```
   (replacing the filename with your own).
3. **Image already present on the server/Colab disk** — simply provide the
   full path to it as the first argument of the function.


In [ ]:
def predict_new_image(image_path, model, class_names, img_size=IMG_SIZE, top_k=3, save_result=True):
    # Loads an image from disk, returns the `top_k` most probable classes,
    # and optionally saves a visualization of the result to predictions/.
    raw_image = tf.io.read_file(image_path)
    image = tf.io.decode_image(raw_image, channels=3, expand_animations=False)
    image = tf.image.resize(image, (img_size, img_size))
    image = tf.cast(image, tf.float32)

    probs = model.predict(tf.expand_dims(image, axis=0), verbose=0)[0]
    top_indices = np.argsort(probs)[::-1][:top_k]

    plt.figure(figsize=(5, 5))
    plt.imshow(image.numpy().astype("uint8"))
    plt.title(f"Prediction: {class_names[top_indices[0]]}\nConfidence: {probs[top_indices[0]] * 100:.1f}%", fontsize=10)
    plt.axis("off")

    if save_result:
        out_name = f"pred_{pathlib.Path(image_path).stem}_{class_names[top_indices[0]]}.png"
        out_path = os.path.join(PREDICTIONS_DIR, out_name)
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        print(f"💾 Result saved: {out_path}")

    plt.show()

    print(f"🔝 Top-{top_k} predictions:")
    for idx in top_indices:
        print(f"   {class_names[idx]:12s} {probs[idx] * 100:5.1f}%")

    return [(class_names[idx], float(probs[idx])) for idx in top_indices]


In [ ]:
# ---------------------------------------------------------------
# Interactive upload and classification of your own image.
# In Google Colab, a file-selection button will appear.
# In other environments (local Jupyter) — call the function manually,
# passing the image path (see the example in the comment below).
# ---------------------------------------------------------------
try:
    from google.colab import files

    print("📤 Choose an image from your computer to classify:")
    uploaded = files.upload()

    for fname in uploaded.keys():
        print(f"\n🖼️ Analyzing file: {fname}")
        result = predict_new_image(fname, loaded_model, loaded_class_names)

except ImportError:
    print("ℹ️ Google Colab environment not detected.")
    print("   Upload an image to the images/ folder, for example, then call:")
    print('   predict_new_image("images/my_photo.jpg", loaded_model, loaded_class_names)')

# Manual call example (uncomment and replace the path):
# result = predict_new_image("images/my_sky.jpg", loaded_model, loaded_class_names)


<a name="podsumowanie"></a>
## 2️⃣8️⃣ Summary

This project built a complete pipeline for classifying weather phenomena from
photographs, using **Transfer Learning**, **Fine Tuning**, and **Grad-CAM** on
the **EfficientNetB0** architecture:

* Loaded and analyzed the **Weather Dataset** (11 classes, ~6862 images),
  including the class distribution and example images for each category.
* Implemented **Data Augmentation** as part of the model's computation graph.
* Split the data into **train / validation / test** sets (70/15/15%)
  ourselves — the original dataset had no predefined split.
* Built the model in two phases: **transfer learning** (frozen base model)
  and **fine-tuning** (last layers unfrozen, low learning rate), with a full
  set of callbacks (`ModelCheckpoint`, `EarlyStopping`, `ReduceLROnPlateau`,
  `TensorBoard`).
* Evaluated the model with multiple metrics: **accuracy, precision, recall,
  F1-score** (macro and weighted), a confusion matrix, and a full per-class
  classification report.
* Implemented **Grad-CAM** to visualize which regions of an image the model
  actually focuses on — a key interpretability tool for real-world model
  deployment.
* Saved the model in `.keras` format along with the class list, verified
  correct loading, and prepared a ready-to-use inference pipeline for new
  images, with automatic result archiving in `predictions/`.

The resulting metrics (accuracy, F1-score) are reported directly in the cells
of Sections 14–20 after training runs — the values depend on the specific
training run and the hardware the notebook is executed on.

<a name="ograniczenia"></a>
## 2️⃣9️⃣ Limitations

* **Visual similarity between some classes.** Phenomena such as `frost` and
  `glaze`, or `snow` and `rime`, can be difficult to distinguish even for a
  human on a single photo — some level of confusion between such pairs is
  practically unavoidable (see Section 19).
* **Small, imbalanced dataset.** ~6862 images across 11 classes is
  relatively little data by deep-learning standards — some classes may be
  underrepresented, which limits the model's generalization ability.
* **No temporal/sequential context.** The model evaluates a single frame —
  it doesn't leverage, for instance, a time sequence of images, which in
  meteorological practice could improve accuracy (weather phenomena evolve
  over time).
* **Internet-sourced images, not a uniform source.** Unlike images from
  weather-monitoring cameras, this dataset contains photos of very varied
  style (amateur, professional, different times of day) — the model may not
  generalize well to footage from one specific, uniform camera.
* **Single phenomenon per image.** The model assumes exactly one label per
  image — it doesn't handle photos depicting multiple co-occurring phenomena
  (e.g. rain and fog at the same time).

<a name="ulepszenia"></a>
## 3️⃣0️⃣ Possible Improvements

* **Larger EfficientNet variants** (B3–B7) or newer architectures (ConvNeXt,
  Vision Transformer) — potentially higher accuracy.
* **More data** — combining with other weather datasets (e.g. WEAPD,
  Multi-class Weather Dataset) to increase the number of examples and
  improve class balance.
* **Multi-label classification** — handling the co-occurrence of several
  phenomena in a single image.
* **Temporal sequence analysis** (e.g. from monitoring cameras) instead of
  single frames — using spatio-temporal models (CNN+LSTM, video
  transformers).
* **Quantization and export to TensorFlow Lite / ONNX** for mobile or
  embedded devices (edge devices at weather stations).
* **Deployment as a REST API** (e.g. FastAPI + Docker), with automatic
  Grad-CAM included in the API response for greater prediction transparency.
* **Active learning** — prioritizing the labeling of new images in regions
  where the model is least confident.
* **Continuous production monitoring** (e.g. using MLflow) — tracking data
  drift and prediction quality over time after deployment.

---

## 🛠️ Tech Stack

`Python` · `TensorFlow / Keras` · `EfficientNetB0` · `Grad-CAM` ·
`scikit-learn` · `NumPy` · `Pandas` · `Matplotlib` · `Seaborn` · `TensorBoard`

## 👤 Author

AIResearchForge — a TensorFlow/Keras-based project for image classification, featuring Transfer Learning, Fine-Tuning, and model explainability using Grad-CAM.

## 📄 License

This project is released under the MIT license (see the `LICENSE` file in the
repository). The Weather Dataset is subject to the license set by its
author — see
[Kaggle — jehanbhathena/weather-dataset](https://www.kaggle.com/datasets/jehanbhathena/weather-dataset).
